### **Удаление выбросов.**

_Считаем данные._

In [1]:
import pandas as pd
import numpy as np

x = pd.read_csv('x_after_lesson_2.csv')
y = pd.read_csv('y_after_lesson_2.csv')

data = pd.concat((x, y), axis=1)

_Узнаем значение квантилей._

In [2]:
#quantile(X) — это значение, ниже которого находится X доля данных.
top_quant = data['log_price_doc'].quantile(0.975)
low_quant = data['log_price_doc'].quantile(0.025)

print(f"Топ 2,5% значение таргета: {top_quant.round(2)}")
print(f"Топ 97,5% значение таргета: {low_quant.round(2)}")

Топ 2,5% значение таргета: 16.7
Топ 97,5% значение таргета: 13.82


_Уберем значения, которые выходят за эти пределы._

In [3]:
#оставим те, что больше нижней и меньше верхней
mask = (data['log_price_doc'] > low_quant) &(data['log_price_doc'] < top_quant)

data = data[mask]

x_new, y_new = data.drop('log_price_doc', axis=1), data['log_price_doc']

_Будем обучать Лассо-регрессию предварительно стандартизировав признаки._

In [4]:
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([('scaler', StandardScaler()), ('Lasso', Lasso(max_iter=100000))])
pipe.fit(x, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('Lasso', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](171,)","['sub_area','ecology','id',...,'month_10','month_11','month_12']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,171
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


_Подберем лучший гиперпарамметр для Лассо-регресии._

In [6]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV

alphas = np.linspace(start=0.01, stop=1, num=30)

new_splitter = TimeSeriesSplit(n_splits=4)

param_grid = {
    "Lasso__alpha": alphas
}


search = GridSearchCV(pipe, param_grid,
                      cv=new_splitter, scoring='neg_mean_squared_error')

search.fit(x_new, y_new)

print(f"Best parameter (CV score={search.best_score_:.5f}):")
print(search.best_params_)

Best parameter (CV score=-0.16734):
{'Lasso__alpha': np.float64(0.04413793103448276)}


_Применим новый параметр и замерим его качество._

In [7]:
from sklearn.model_selection import cross_validate
pipe.set_params(Lasso__alpha=search.best_params_['Lasso__alpha'])

cv_result_pipe = cross_validate(pipe, x_new, y_new,
                                scoring='neg_mean_squared_error',
                                cv=new_splitter, return_train_score=True)

print(f"Среднее MSLE на тренировочных фолдах: {-np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Среднее MSLE на тестовых фолдах: {-np.mean(cv_result_pipe['test_score']).round(3)}")

Среднее MSLE на тренировочных фолдах: 0.157
Среднее MSLE на тестовых фолдах: 0.167


### **Сегментация данных.**